In [1]:
from dotenv import load_dotenv
import os

load_dotenv('db.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

In [2]:
import torch

In [3]:
pcst_output = torch.load('stark_qa_v17/processed/test_data_nodes.pt')

/var/folders/rp/zlbqb_1d7j53cg1t8g_qq8q40000gn/T/ipykernel_83971/4185301223.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pcst_output = torch.load('stark_qa_v17/proces

In [4]:
pcst_output.keys()

dict_keys(['correct_nodes', 'topk_nodes', 'subgraph1_nodes', 'pcst_nodes'])

In [5]:
from stark_qa import load_qa

In [6]:
qa_dataset = load_qa("prime")
qa_raw_test = qa_dataset.get_subset('test')

Use file from /Users/sbr/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/qa/prime/stark_qa/stark_qa_human_generated_eval.csv.


In [7]:
query_embedding_dict = torch.load('data-loading/emb/prime/text-embedding-ada-002/query/query_emb_dict.pt')

In [8]:
ordered_pcst_nodes_20 = {}

In [9]:
from neo4j import Driver, GraphDatabase

In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
    # loop over key values of pcst_output['pcst_nodes'] dict
    for query_id, pcst_nodes in pcst_output['pcst_nodes'].items():
        # get the query text
        query_emb = query_embedding_dict[query_id]

        res = driver.execute_query("""
        UNWIND $nodeIds AS nodeId
        MATCH (node:_Entity_ {nodeId:nodeId}) RETURN node.nodeId as nodeId, node.textEmbedding AS textEmbedding
        """,
                                   parameters_={
                                       "nodeIds": pcst_nodes})
        node_embs = pd.DataFrame([rec.data() for rec in res.records])
        embeddings = np.vstack(node_embs['textEmbedding'].values)
        cos_sim = cosine_similarity(embeddings, query_emb.reshape(1,-1)).ravel()
        top_n_indices = np.argsort(cos_sim)[-20:][::-1]
        top_n_nodeIds = node_embs.iloc[top_n_indices]['nodeId'].to_numpy()

        breakpoint()
        ordered_pcst_nodes_20[query_id] = top_n_nodeIds

In [12]:
len(ordered_pcst_nodes_20)

2801

In [ ]:
ordered_pcst_nodes_20[10630]

In [16]:
from compute_metrics import compute_intermediate_metrics

In [17]:
pcst_output.keys()

dict_keys(['correct_nodes', 'topk_nodes', 'subgraph1_nodes', 'pcst_nodes'])

In [18]:
len(pcst_output['correct_nodes'])

2801

In [19]:
compute_intermediate_metrics(pcst_output['correct_nodes'], ordered_pcst_nodes_20)

F1:              0.059965080657517315
Precision:       0.03570153516601214
Recall:          0.35934884544441004
Exact hit@1:     0.12602641913602286
Exact hit@5:     0.3156015708675473
Exact hit@any:   0.4423420207068904
Recall@20:       0.35934884544441004
MRR:             0.20963870057779188
Num predictions: 20.0


In [20]:
eval_outs = torch.load('stark_qa_v17/models/v17_gnn-llm-llama3.1-8b_eval_outs.pt')

In [21]:
evals = pd.concat([pd.DataFrame(d) for d in eval_outs])

In [ ]:
evals

In [ ]:
len(qa_raw_test.indices)

In [24]:
i = 0
node_pred_from_text = {}
for pred, label in zip(evals.pred.tolist(), evals.label.tolist()):
    pred = pred.split('[/s]')[0].strip().split('|')
    # covert every string to all uppercase
    pred = [p.upper() for p in pred]
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as driver:
        res = driver.execute_query("""
        MATCH (n) WHERE toUpper(n.name) IN $pred RETURN n.nodeId as nodeId
        """,
                                   parameters_={
                                       "pred": pred})
        pred_ids = [rec.data() for rec in res.records]
        node_ids = [item['nodeId'] for item in pred_ids]
        node_pred_from_text[i] = node_ids
        i += 1

In [ ]:
node_pred_from_text

In [33]:
consecutive_correct_nodes = {i:v for i,v in enumerate(pcst_output['correct_nodes'].values())}

In [ ]:
consecutive_correct_nodes

In [ ]:
compute_intermediate_metrics(consecutive_correct_nodes, node_pred_from_text)

In [ ]:
#consecutive_correct_nodes, node_pred_from_text, ordered_pcst_nodes_20

In [ ]:
ordered_pcst_nodes_20

In [37]:
ordered_pcst_nodes_20_reindex = {i: v.tolist() for i, v in enumerate(ordered_pcst_nodes_20.values())}

In [ ]:
ordered_pcst_nodes_20_reindex

In [39]:
merged_pred = {k: node_pred_from_text[k] + ordered_pcst_nodes_20_reindex[k] for k in node_pred_from_text}

In [41]:
ensemble_pred = {k: list(dict.fromkeys(node_pred_from_text[k] + ordered_pcst_nodes_20_reindex[k]))[:20] for k in node_pred_from_text}

In [ ]:
compute_intermediate_metrics(consecutive_correct_nodes, ensemble_pred)

In [ ]:
pcst_output['correct_nodes']